In [4]:
#  Step 1: Import Libraries
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#  Step 2: Data Loading
data = pd.read_csv("tsla.csv")

#  Step 3: Data Exploration
print("First 5 rows:\n", data.head())
print("\nDataset Info:\n")
data.info()
print("\nStatistical Summary:\n", data.describe())

# Step 4: Handle Missing Values
# Fill numeric columns with mean
data.fillna(data.mean(numeric_only=True), inplace=True)

# Step 5: Convert Date (Categorical → Numerical)
data['Date'] = pd.to_datetime(data['Date'])

# Extract useful features from Date
data['Year'] = data['Date'].dt.year
data['Month'] = data['Date'].dt.month
data['Day'] = data['Date'].dt.day

# Drop original Date column
data.drop('Date', axis=1, inplace=True)

#  Step 6: Feature Engineering (Important for TSLA)
data['Prev_Close'] = data['Close'].shift(1)
data['MA_5'] = data['Close'].rolling(5).mean()

# Drop rows with NaN (created by lag/rolling)
data.dropna(inplace=True)

#  Step 7: Feature Selection
X = data.drop('Close', axis=1)   # Features
y = data['Close']                # Target

#  Step 8: Train-Test Split (No shuffle for time series)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

#  Step 9: Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#  Step 10: Model Training
model = LinearRegression()
model.fit(X_train, y_train)

#  Step 11: Prediction
y_pred = model.predict(X_test)

#  Step 12: Evaluation
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation:")
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

#  Step 13: Prediction on New Data
# Take last row as example
new_data = X.iloc[-1:].copy()

# Scale
new_data_scaled = scaler.transform(new_data)

prediction = model.predict(new_data_scaled)
print("\nPredicted Next Close Price:", prediction[0])

#  Step 14: Model Saving
pickle.dump(model, open("tsla_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))

print("\nModel and scaler saved successfully!")

First 5 rows:
          Date      Open      High       Low     Close     Volume
0  2010-06-29  1.266667  1.666667  1.169333  1.592667  281494500
1  2010-06-30  1.719333  2.028000  1.553333  1.588667  257806500
2  2010-07-01  1.666667  1.728000  1.351333  1.464000  123282000
3  2010-07-02  1.533333  1.540000  1.247333  1.280000   77097000
4  2010-07-06  1.333333  1.333333  1.055333  1.074000  103003500

Dataset Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3948 entries, 0 to 3947
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    3948 non-null   object 
 1   Open    3948 non-null   float64
 2   High    3948 non-null   float64
 3   Low     3948 non-null   float64
 4   Close   3948 non-null   float64
 5   Volume  3948 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 185.2+ KB

Statistical Summary:
               Open         High          Low        Close        Volume
count  3948.000000